# UAV Path Prediction

In [ ]:
import pandas as pd
import numpy as np
import glob
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

# Configure TensorFlow to use the M4 GPU (Apple Metal / MPS)
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Apple Silicon GPU Active: {gpus[0].name}")
    except RuntimeError as e:
        print(e)
else:
    print(" No GPU detected. Defaulting to CPU.")
    

In [ ]:
files = sorted(glob.glob("dataset/*.csv"))
test_files1 = sorted(glob.glob("synthetic_uavs/*.csv"))
test_files2 = sorted(glob.glob("data/uav_synthetic_output/csv/*.csv"))

train_files = files[:9]
test_file_1   = test_files1
test_file_2 = test_files2

features  = ["lat", "lon", "height", "Omega", "Kappa", "Phi1", "Phi2"]
date_col  = "date"
SEQ_LEN   = 20    # reduced from 40 — shorter window reduces lag
VAL_RATIO = 0.15

In [ ]:
def load_sorted(filepath):
    df = pd.read_csv(filepath)
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).reset_index(drop=True)
    return df[features].values.astype(np.float32)

In [ ]:
def add_velocity_features(data):
    """
    Append velocity (step-to-step delta) for lat, lon, height
    to each row.  Row 0 gets velocity = 0 (no prior step).

    Input shape : (T, 7)   [lat, lon, height, Omega, Kappa, Phi1, Phi2]
    Output shape: (T, 10)  [original 7 + vel_lat, vel_lon, vel_height]
    """
    velocity        = np.zeros_like(data[:, :3])          # (T, 3)
    velocity[1:]    = data[1:, :3] - data[:-1, :3]        # delta from t-1 to t
    return np.concatenate([data, velocity], axis=1).astype(np.float32)

In [ ]:
def make_sequences_delta(data_with_vel, raw_data, seq_len):
    """
    X : window of seq_len steps using enriched features (10 cols)
    y : delta lat / lon / height at the next step  ← now 3 outputs
    """
    X, y = [], []
    for i in range(len(data_with_vel) - seq_len):
        X.append(data_with_vel[i : i + seq_len])
        delta = raw_data[i + seq_len, :3] - raw_data[i + seq_len - 1, :3]  # lat, lon, height
        y.append(delta)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


In [ ]:
# ── Fit scaler on training rows only (raw 7 features + 3 velocity = 10) ──
all_train_rows = []
for f in train_files:
    raw   = load_sorted(f)
    split = int(len(raw) * (1 - VAL_RATIO))
    enriched = add_velocity_features(raw[:split])
    all_train_rows.append(enriched)

all_train_rows = np.concatenate(all_train_rows, axis=0)
x_scaler = StandardScaler()
x_scaler.fit(all_train_rows)

n_features = all_train_rows.shape[1]   # 10
print(f"Features per timestep: {n_features}  (7 original + 3 velocity)")
print(f"x_scaler fitted on {all_train_rows.shape[0]} rows")

In [ ]:
X_train_list, y_train_list = [], []
X_val_list,   y_val_list   = [], []

for f in train_files:
    raw   = load_sorted(f)
    split = int(len(raw) * (1 - VAL_RATIO))

    for raw_split, X_list, y_list in [
        (raw[:split], X_train_list, y_train_list),
        (raw[split:], X_val_list,   y_val_list)
    ]:
        if len(raw_split) <= SEQ_LEN:
            continue
        enriched = add_velocity_features(raw_split)
        Xs, ys   = make_sequences_delta(enriched, raw_split, SEQ_LEN)
        Xs_scaled = x_scaler.transform(
                        Xs.reshape(-1, n_features)
                    ).reshape(Xs.shape)
        X_list.append(Xs_scaled)
        y_list.append(ys)

X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)
X_val   = np.concatenate(X_val_list,   axis=0)
y_val   = np.concatenate(y_val_list,   axis=0)

print(f"X_train: {X_train.shape}   y_train: {y_train.shape}")
print(f"X_val  : {X_val.shape}     y_val  : {y_val.shape}")
print(f"\nDelta stats — lat    mean={y_train[:,0].mean():.6f}  std={y_train[:,0].std():.6f}")
print(f"Delta stats — lon    mean={y_train[:,1].mean():.6f}  std={y_train[:,1].std():.6f}")
print(f"Delta stats — height mean={y_train[:,2].mean():.6f}  std={y_train[:,2].std():.6f}")


In [ ]:
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train)
y_val_scaled   = y_scaler.transform(y_val)

In [ ]:
with tf.device('/GPU:0'):
    model = Sequential([
        Input(shape=(SEQ_LEN, n_features)),
        LSTM(128, return_sequences=True),
        Dropout(0.2),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dense(32, activation='relu'),
        Dense(3)   # delta_lat, delta_lon, delta_height
    ])

    model.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=['mae']
    )
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15,
                  restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_uav_model.keras', monitor='val_loss',
                    save_best_only=True, verbose=0)
]

# Force full dataset fitting onto M4 GPU
with tf.device('/GPU:0'):
    history = model.fit(
        X_train, y_train_scaled,
        validation_data=(X_val, y_val_scaled),
        epochs=150,
        batch_size=32,
        callbacks=callbacks,
        verbose=1
    )

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'],     label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Huber Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.grid()
plt.tight_layout()
plt.show()

In [ ]:
# ── Test_1 ──
for i, f in enumerate(test_file_1):
    test_raw      = load_sorted(f)
    test_enriched = add_velocity_features(test_raw)

    X_test, y_test_delta = make_sequences_delta(test_enriched, test_raw, SEQ_LEN)
    X_test_scaled = x_scaler.transform(
                        X_test.reshape(-1, n_features)
                    ).reshape(X_test.shape)

    pred_delta_scaled = model.predict(X_test_scaled)
    pred_delta        = y_scaler.inverse_transform(pred_delta_scaled)  # (T', 3)

    # Reconstruct absolute position: anchor (last known) + predicted delta
    anchor = test_raw[SEQ_LEN - 1 : SEQ_LEN - 1 + len(pred_delta), :3]  # lat, lon, height
    y_pred = anchor + pred_delta                                           # (T', 3)
    actual = test_raw[SEQ_LEN:, :3]                                       # (T', 3)

    # ── A4-optimised figure: 2 subplots only (no error plot) ──
    fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

    # ── Subplot 1: Trajectory (Lat / Lon) ──
    axes[0].plot(actual[:,1], actual[:,0],       label='Actual',    linewidth=2.5)
    axes[0].plot(y_pred[:,1], y_pred[:,0], '--', label='Predicted', linewidth=2.0, color='red')
    axes[0].set_xlabel('Longitude', fontsize=13)
    axes[0].set_ylabel('Latitude',  fontsize=13)
    axes[0].set_title(f'UAV {i} — Trajectory (Lat/Lon)', fontsize=16, fontweight='bold')
    axes[0].legend(fontsize=12)
    axes[0].tick_params(labelsize=11)
    axes[0].grid(alpha=0.4)

    # ── Subplot 2: Height over Time ──
    axes[1].plot(actual[:,2],       label='Actual Height',    linewidth=2.0)
    axes[1].plot(y_pred[:,2], '--', label='Predicted Height', linewidth=2.0, color='red')
    axes[1].set_xlabel('Timestep', fontsize=13)
    axes[1].set_ylabel('Height',   fontsize=13)
    axes[1].set_title(f'UAV {i} — Height Prediction', fontsize=16, fontweight='bold')
    axes[1].legend(fontsize=12)
    axes[1].tick_params(labelsize=11)
    axes[1].grid(alpha=0.4)

    plt.tight_layout(pad=2.0)
    plt.savefig(f'predicted_path_image/uav_test1_predicted_path_UAV_{i}.png',
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()


In [ ]:
# # ── Test_2 ──
# for i, f in enumerate(test_file_2):
#     test_raw      = load_sorted(f)
#     test_enriched = add_velocity_features(test_raw)

#     X_test, y_test_delta = make_sequences_delta(test_enriched, test_raw, SEQ_LEN)
#     X_test_scaled = x_scaler.transform(
#                         X_test.reshape(-1, n_features)
#                     ).reshape(X_test.shape)

#     pred_delta_scaled = model.predict(X_test_scaled)
#     pred_delta        = y_scaler.inverse_transform(pred_delta_scaled)  # (T', 3)

#     # Reconstruct absolute position: anchor (last known) + predicted delta
#     anchor = test_raw[SEQ_LEN - 1 : SEQ_LEN - 1 + len(pred_delta), :3]  # lat, lon, height
#     y_pred = anchor + pred_delta                                           # (T', 3)
#     actual = test_raw[SEQ_LEN:, :3]                                       # (T', 3)

#     # ── A4-optimised figure: 2 subplots only (no error plot) ──
#     fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

#     # ── Subplot 1: Trajectory (Lat / Lon) ──
#     axes[0].plot(actual[:,1], actual[:,0],       label='Actual',    linewidth=2.5)
#     axes[0].plot(y_pred[:,1], y_pred[:,0], '--', label='Predicted', linewidth=2.0, color='red')
#     axes[0].set_xlabel('Longitude', fontsize=13)
#     axes[0].set_ylabel('Latitude',  fontsize=13)
#     axes[0].set_title(f'UAV {i} — Trajectory (Lat/Lon)', fontsize=16, fontweight='bold')
#     axes[0].legend(fontsize=12)
#     axes[0].tick_params(labelsize=11)
#     axes[0].grid(alpha=0.4)

#     # ── Subplot 2: Height over Time ──
#     axes[1].plot(actual[:,2],       label='Actual Height',    linewidth=2.0)
#     axes[1].plot(y_pred[:,2], '--', label='Predicted Height', linewidth=2.0, color='red')
#     axes[1].set_xlabel('Timestep', fontsize=13)
#     axes[1].set_ylabel('Height',   fontsize=13)
#     axes[1].set_title(f'UAV {i} — Height Prediction', fontsize=16, fontweight='bold')
#     axes[1].legend(fontsize=12)
#     axes[1].tick_params(labelsize=11)
#     axes[1].grid(alpha=0.4)

#     plt.tight_layout(pad=2.0)
#     plt.savefig(f'uav_test2_predicted_path_UAV_{i}.png',
#                 dpi=300, bbox_inches='tight', facecolor='white')
#     plt.show()


In [ ]:
# mae  = mean_absolute_error(actual, y_pred)
# rmse = np.sqrt(mean_squared_error(actual, y_pred))

# lat_mae = mean_absolute_error(actual[:,0], y_pred[:,0])
# lon_mae = mean_absolute_error(actual[:,1], y_pred[:,1])

# print(f"Overall MAE  : {mae:.6f} degrees")
# print(f"Overall RMSE : {rmse:.6f} degrees")
# print(f"  Lat MAE    : {lat_mae:.6f}  (~{lat_mae * 111000:.1f} m)")
# print(f"  Lon MAE    : {lon_mae:.6f}  (~{lon_mae * 111000:.1f} m)")

In [ ]:
# ── Average Prediction Error per UAV — Test File 1 ──
avg_errors = []
file_names = []

for i, f in enumerate(test_file_1):
    try:
        test_raw      = load_sorted(f)
        test_enriched = add_velocity_features(test_raw)

        X_test, y_test_delta = make_sequences_delta(test_enriched, test_raw, SEQ_LEN)
        if len(X_test) == 0:
            continue

        X_test_scaled = x_scaler.transform(
                            X_test.reshape(-1, n_features)
                        ).reshape(X_test.shape)

        pred_delta_scaled = model.predict(X_test_scaled, verbose=0)
        pred_delta        = y_scaler.inverse_transform(pred_delta_scaled)  # (T', 3)

        anchor = test_raw[SEQ_LEN - 1 : SEQ_LEN - 1 + len(pred_delta), :3]
        y_pred = anchor + pred_delta
        actual = test_raw[SEQ_LEN:, :3]

        # 3D Euclidean error per timestep, then average
        errors = np.sqrt(((actual - y_pred) ** 2).sum(axis=1))
        avg_errors.append(errors.mean())
        file_names.append(f"UAV {i+1}")

    except Exception as e:
        print(f"Skipping {f}: {e}")
        continue

avg_errors  = np.array(avg_errors)
uav_indices = np.arange(1, len(avg_errors) + 1)

# ── A4-optimised figure: Rolling Average Trend Line only ──
window      = 10
rolling_avg = pd.Series(avg_errors).rolling(window, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 6), dpi=300)

ax.plot(uav_indices, avg_errors,  color='steelblue', alpha=0.35,
        linewidth=0.9, label='Per-UAV Error')
ax.plot(uav_indices, rolling_avg, color='steelblue', linewidth=2.5,
        label=f'Rolling Avg  (window={window})')
ax.fill_between(uav_indices,
                avg_errors - avg_errors.std(),
                avg_errors + avg_errors.std(),
                alpha=0.15, color='steelblue', label='±1 Std Dev Band')
ax.axhline(avg_errors.mean(), color='blue', linestyle='--', linewidth=2.0,
           label=f'Overall Mean: {avg_errors.mean():.5f}')

ax.set_xlabel('UAV Index',              fontsize=13)
ax.set_ylabel('Avg 3D Euclidean Error', fontsize=13)
ax.set_title('Average 3D Prediction Error per UAV — Test File 1 — Rolling Avg Trend',
             fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
ax.tick_params(labelsize=11)
ax.set_xlim(0, len(avg_errors) + 1)
ax.grid(alpha=0.4)

plt.tight_layout(pad=2.0)
plt.savefig('avg_pred_error_trend_test1.png',
            dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

print(f"\n{'='*45}")
print(f"  Total UAVs evaluated : {len(avg_errors)}")
print(f"  Mean 3D error        : {avg_errors.mean():.6f}")
print(f"  Median 3D error      : {np.median(avg_errors):.6f}")
print(f"  Std deviation        : {avg_errors.std():.6f}")
print(f"  Min error  (UAV {np.argmin(avg_errors)+1:>3d}) : {avg_errors.min():.6f}")
print(f"  Max error  (UAV {np.argmax(avg_errors)+1:>3d}) : {avg_errors.max():.6f}")
print(f"{'='*45}")


In [ ]:
# # ── Average Prediction Error per UAV — Test File 2 ──
# avg_errors = []
# file_names = []

# for i, f in enumerate(test_file_2):
#     try:
#         test_raw      = load_sorted(f)
#         test_enriched = add_velocity_features(test_raw)

#         X_test, y_test_delta = make_sequences_delta(test_enriched, test_raw, SEQ_LEN)
#         if len(X_test) == 0:
#             continue

#         X_test_scaled = x_scaler.transform(
#                             X_test.reshape(-1, n_features)
#                         ).reshape(X_test.shape)

#         pred_delta_scaled = model.predict(X_test_scaled, verbose=0)
#         pred_delta        = y_scaler.inverse_transform(pred_delta_scaled)  # (T', 3)

#         anchor = test_raw[SEQ_LEN - 1 : SEQ_LEN - 1 + len(pred_delta), :3]
#         y_pred = anchor + pred_delta
#         actual = test_raw[SEQ_LEN:, :3]

#         # 3D Euclidean error per timestep, then average
#         errors = np.sqrt(((actual - y_pred) ** 2).sum(axis=1))
#         avg_errors.append(errors.mean())
#         file_names.append(f"UAV {i+1}")

#     except Exception as e:
#         print(f"Skipping {f}: {e}")
#         continue

# avg_errors  = np.array(avg_errors)
# uav_indices = np.arange(1, len(avg_errors) + 1)

# # ── A4-optimised figure: Rolling Average Trend Line only ──
# window      = 10
# rolling_avg = pd.Series(avg_errors).rolling(window, center=True).mean()

# fig, ax = plt.subplots(figsize=(14, 6), dpi=300)

# ax.plot(uav_indices, avg_errors,  color='steelblue', alpha=0.35,
#         linewidth=0.9, label='Per-UAV Error')
# ax.plot(uav_indices, rolling_avg, color='steelblue', linewidth=2.5,
#         label=f'Rolling Avg  (window={window})')
# ax.fill_between(uav_indices,
#                 avg_errors - avg_errors.std(),
#                 avg_errors + avg_errors.std(),
#                 alpha=0.15, color='steelblue', label='±1 Std Dev Band')
# ax.axhline(avg_errors.mean(), color='blue', linestyle='--', linewidth=2.0,
#            label=f'Overall Mean: {avg_errors.mean():.5f}')

# ax.set_xlabel('UAV Index',              fontsize=13)
# ax.set_ylabel('Avg 3D Euclidean Error', fontsize=13)
# ax.set_title('Average 3D Prediction Error per UAV — Test File 2 — Rolling Avg Trend',
#              fontsize=16, fontweight='bold')
# ax.legend(fontsize=12)
# ax.tick_params(labelsize=11)
# ax.set_xlim(0, len(avg_errors) + 1)
# ax.grid(alpha=0.4)

# plt.tight_layout(pad=2.0)
# plt.savefig('avg_pred_error_trend_test2.png',
#             dpi=300, bbox_inches='tight', facecolor='white')
# plt.show()

# print(f"\n{'='*45}")
# print(f"  Total UAVs evaluated : {len(avg_errors)}")
# print(f"  Mean 3D error        : {avg_errors.mean():.6f}")
# print(f"  Median 3D error      : {np.median(avg_errors):.6f}")
# print(f"  Std deviation        : {avg_errors.std():.6f}")
# print(f"  Min error  (UAV {np.argmin(avg_errors)+1:>3d}) : {avg_errors.min():.6f}")
# print(f"  Max error  (UAV {np.argmax(avg_errors)+1:>3d}) : {avg_errors.max():.6f}")
# print(f"{'='*45}")


## Dynamic Fixed-K Clustering via K-Means
At every timestamp the LSTM-predicted `(lat, lon, height)` of all 500 UAVs is clustered
using **K-Means with a fixed K** chosen by the Elbow + Silhouette method.

**LSTM now predicts all 3 spatial dimensions:**
- `Δlat`, `Δlon`, `Δheight` are all predicted as deltas and reconstructed to absolute positions.
- No ground-truth height leakage — clustering uses fully predicted 3D positions.

**Why fixed K over DBSCAN?**
- Routing table always has exactly K Cluster Heads — no surprise topology changes
- Aligns with standard UAV swarm protocols (LEACH, HEED, PEGASIS)
- Clean, comparable metrics across all timestamps

**Pipeline:**
1. Collect LSTM-predicted 3D positions (lat, lon, height) for all 500 UAVs
2. Align to a common time grid  →  `(T × 500 × 3)` matrix
3. Find optimal K via Elbow + Silhouette (run once on a sample of timestamps)
4. Apply fixed K-Means at every timestamp + elect Cluster Heads
5. Visualise cluster dynamics
6. Export results to CSV


In [ ]:
# ── Step 1: Collect LSTM-predicted 3D positions for all 500 UAVs ──
# The LSTM now predicts (Δlat, Δlon, Δheight).
# Absolute positions are reconstructed as: anchor + predicted_delta  for all 3 dims.
# No actual/ground-truth values are used here — fully predicted 3D positions.
# Output: uav_predicted_positions[uav_index] → ndarray (T_i, 3)

import time

uav_predicted_positions = {}   

# Explicitly perform prediction on GPU
with tf.device('/GPU:0'):
    for i, f in enumerate(test_file_2):
        try:
            test_raw      = load_sorted(f)
            test_enriched = add_velocity_features(test_raw)

            X_test, _ = make_sequences_delta(test_enriched, test_raw, SEQ_LEN)
            if len(X_test) == 0:
                continue

            X_test_scaled = x_scaler.transform(
                X_test.reshape(-1, n_features)
            ).reshape(X_test.shape)

            pred_delta_scaled = model.predict(X_test_scaled, verbose=0)
            pred_delta        = y_scaler.inverse_transform(pred_delta_scaled)  

            anchor = test_raw[SEQ_LEN - 1 : SEQ_LEN - 1 + len(pred_delta), :3]  
            uav_predicted_positions[i] = anchor + pred_delta                     

        except Exception as e:
            print(f"Skipping UAV {i+1}: {e}")

print(f"Collected fully-predicted 3D positions for {len(uav_predicted_positions)} UAVs")

# Measure empirical GPU latency [X] for a batch of 500 UAVs
simulated_batch = np.zeros((500, SEQ_LEN, n_features), dtype=np.float32)

with tf.device('/GPU:0'):
    _ = model.predict(simulated_batch, verbose=0) # Warm-up
    
    lstm_times_ms = []
    for _ in range(30):
        start_time = time.perf_counter()
        _ = model.predict(simulated_batch, verbose=0)
        end_time = time.perf_counter()
        lstm_times_ms.append((end_time - start_time) * 1000)

avg_lstm_ms = np.mean(lstm_times_ms)
print(f"\n[X] GPU LSTM Batch Prediction Latency (500 UAVs): {avg_lstm_ms:.2f} ms")

In [ ]:
# ── Step 2: Align UAVs to a common time grid ──
# Truncate every UAV to the shortest prediction horizon → shared T timesteps.

min_timesteps = min(pos.shape[0] for pos in uav_predicted_positions.values())
uav_ids       = sorted(uav_predicted_positions.keys())
n_uavs        = len(uav_ids)

# Aligned 3-D matrix → (T, N_uavs, 3)
position_matrix = np.stack(
    [uav_predicted_positions[i][:min_timesteps] for i in uav_ids],
    axis=1
)

print(f"Common time grid  : {min_timesteps} timesteps")
print(f"UAVs aligned      : {n_uavs}")
print(f"Position matrix   : {position_matrix.shape}   (timesteps × uavs × [lat,lon,height])")


In [ ]:
# ── Step 3: Find Optimal K via Elbow + Silhouette ──
#
# Strategy: sample N_SAMPLE_TIMESTAMPS evenly spread timestamps,
# run K-Means for K = 2..MAX_K on each, average the inertia and
# silhouette scores, then plot both curves to identify the best K.
#
# Rule of thumb for 500 UAVs:
#   Each cluster should carry ~20-40 UAVs  →  K ≈ 13–25 is a reasonable range.

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler as PositionScaler

MAX_K              = 30          # upper bound on K to evaluate
N_SAMPLE_TIMESTAMPS = 10         # number of timestamps to average over

sample_ts = np.linspace(0, min_timesteps - 1, N_SAMPLE_TIMESTAMPS, dtype=int)

inertias    = []
silhouettes = []
K_range     = range(2, MAX_K + 1)

print("Running Elbow + Silhouette analysis ...")
for K in K_range:
    inertia_k  = []
    sil_k      = []
    for t in sample_ts:
        pos_t   = position_matrix[t]                          # (n_uavs, 3)
        pos_sc  = PositionScaler().fit_transform(pos_t)
        km      = KMeans(n_clusters=K, random_state=42, n_init=10)
        labels  = km.fit_predict(pos_sc)
        inertia_k.append(km.inertia_)
        if len(set(labels)) > 1:                              # silhouette needs ≥2 clusters
            sil_k.append(silhouette_score(pos_sc, labels, sample_size=min(500, n_uavs)))
    inertias.append(np.mean(inertia_k))
    silhouettes.append(np.mean(sil_k) if sil_k else 0)
    print(f"  K={K:2d}  inertia={inertias[-1]:8.1f}  silhouette={silhouettes[-1]:.4f}")

inertias    = np.array(inertias)
silhouettes = np.array(silhouettes)

# ── Plot ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(K_range), inertias, 'o-', color='royalblue', linewidth=2)
axes[0].set_title('Elbow Method — Inertia vs K', fontsize=13)
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Average Inertia (lower = tighter clusters)')
axes[0].grid(alpha=0.4)

axes[1].plot(list(K_range), silhouettes, 's-', color='seagreen', linewidth=2)
axes[1].set_title('Silhouette Score vs K', fontsize=13)
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Avg Silhouette Score (higher = better separation)')
axes[1].grid(alpha=0.4)

# Mark the best silhouette K
best_sil_k = list(K_range)[np.argmax(silhouettes)]
axes[1].axvline(best_sil_k, color='red', linestyle='--', linewidth=1.5,
                label=f'Best silhouette K={best_sil_k}')
axes[1].legend(fontsize=11)

plt.suptitle('Optimal K Selection — Averaged over Sampled Timestamps', fontsize=14)
plt.tight_layout()
plt.savefig('optimal_k_selection.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nBest K by silhouette score : {best_sil_k}")
print(f"Rule-of-thumb (500/25)     : 20")
print(f"\n→ Set OPTIMAL_K below. Default = best_sil_k.")


In [ ]:
# ── Step 4: Apply Fixed K-Means at Every Timestamp ──
#
# OPTIMAL_K is set to the best silhouette K found above.
# You can override it manually if the elbow suggests a different value.
#
# At each timestamp:
#   1. Normalise UAV positions (lat, lon, height)
#   2. Run K-Means with OPTIMAL_K clusters
#   3. Elect Cluster Head = UAV closest to its cluster centroid
import time

# ── Step 4: Apply Fixed K-Means at Every Timestamp ──

OPTIMAL_K = best_sil_k   # ← override here if desired, e.g. OPTIMAL_K = 20

print(f"Using fixed K = {OPTIMAL_K} clusters across all {min_timesteps} timestamps")

cluster_labels_all  = []   # (T, n_uavs) — cluster id per UAV per timestep
cluster_heads_all   = []   # list of dicts: { cluster_id : uav_global_index }
inertia_per_t       = []   # K-Means inertia at each timestep (cluster tightness)
clustering_times_ms = []   # Timing list for latency measurement

for t in range(min_timesteps):
    pos_t  = position_matrix[t]                              # (n_uavs, 3)
    
    start_time = time.perf_counter()                        # START TIMER [Y]
    pos_sc = PositionScaler().fit_transform(pos_t)
    km     = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=5)
    labels = km.fit_predict(pos_sc)
    end_time = time.perf_counter()                          # END TIMER [Y]
    
    clustering_times_ms.append((end_time - start_time) * 1000)
    inertia_per_t.append(km.inertia_)
    cluster_labels_all.append(labels)

    # ── Cluster Head Election ──
    # For every cluster, elect the UAV closest to the centroid (in original space).
    heads = {}
    for cid in range(OPTIMAL_K):
        members  = np.where(labels == cid)[0]
        centroid = pos_t[members].mean(axis=0)
        dists    = np.linalg.norm(pos_t[members] - centroid, axis=1)
        heads[cid] = uav_ids[members[np.argmin(dists)]]      # store global UAV index
    cluster_heads_all.append(heads)

cluster_labels_all = np.array(cluster_labels_all)            # (T, n_uavs)
inertia_per_t      = np.array(inertia_per_t)

# Cluster sizes at every timestamp  →  (T, K)
cluster_sizes_all = np.array([
    [np.sum(cluster_labels_all[t] == k) for k in range(OPTIMAL_K)]
    for t in range(min_timesteps)
])

# Compute average latency [Y]
avg_kmeans_ms = np.mean(clustering_times_ms)

print(f"\nClustering complete.")
print(f"  Fixed K                         : {OPTIMAL_K}")
print(f"  Avg cluster size                : {cluster_sizes_all.mean():.1f} UAVs")
print(f"  Min cluster size (over all t,k) : {cluster_sizes_all.min()}")
print(f"  Max cluster size (over all t,k) : {cluster_sizes_all.max()}")
print(f"  Avg inertia per timestamp       : {inertia_per_t.mean():.2f}")

# Latency outputs
print(f"\n[Y] K-Means Clustering Latency per timestep: {avg_kmeans_ms:.2f} ms")
if 'avg_lstm_ms' in locals() or 'avg_lstm_ms' in globals():
    print(f"Total Computational Overhead [X + Y]: {avg_lstm_ms + avg_kmeans_ms:.2f} ms")

In [ ]:
# ── Step 5a: Cluster Dynamics Over Time ──

fig, axes = plt.subplots(3, 1, figsize=(16, 14))

# Top: inertia (cluster tightness) over time
axes[0].plot(inertia_per_t, color='royalblue', linewidth=1.5)
axes[0].fill_between(range(min_timesteps), inertia_per_t, alpha=0.15, color='royalblue')
axes[0].axhline(inertia_per_t.mean(), color='navy', linestyle='--', linewidth=1.5,
                label=f'Mean inertia: {inertia_per_t.mean():.1f}')
axes[0].set_title(f'K-Means Inertia Over Time  (K={OPTIMAL_K}, lower = tighter clusters)', fontsize=14)
axes[0].set_xlabel('Timestamp')
axes[0].set_ylabel('Inertia')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.4)

# Middle: cluster size spread over time (min / mean / max bands)
size_min  = cluster_sizes_all.min(axis=1)
size_mean = cluster_sizes_all.mean(axis=1)
size_max  = cluster_sizes_all.max(axis=1)
axes[1].plot(size_mean, color='seagreen', linewidth=2, label='Mean cluster size')
axes[1].fill_between(range(min_timesteps), size_min, size_max,
                     alpha=0.2, color='seagreen', label='Min–Max range')
axes[1].axhline(n_uavs / OPTIMAL_K, color='darkgreen', linestyle='--', linewidth=1.5,
                label=f'Ideal size ({n_uavs}/{OPTIMAL_K} = {n_uavs//OPTIMAL_K})')
axes[1].set_title('Cluster Size Spread Over Time  (Min / Mean / Max)', fontsize=14)
axes[1].set_xlabel('Timestamp')
axes[1].set_ylabel('UAVs in Cluster')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.4)

# Bottom: cluster size distribution at midpoint
t_mid      = min_timesteps // 2
sizes_mid  = cluster_sizes_all[t_mid]
bar_colors = plt.cm.tab20(np.linspace(0, 1, OPTIMAL_K))
axes[2].bar(range(OPTIMAL_K), sizes_mid, color=bar_colors, edgecolor='none')
axes[2].axhline(n_uavs / OPTIMAL_K, color='black', linestyle='--', linewidth=1.5,
                label=f'Ideal = {n_uavs // OPTIMAL_K} UAVs')
axes[2].set_title(f'Cluster Size Distribution at Midpoint (t={t_mid})', fontsize=14)
axes[2].set_xlabel('Cluster ID')
axes[2].set_ylabel('Number of UAVs')
axes[2].legend(fontsize=11)
axes[2].grid(axis='y', alpha=0.4)

plt.tight_layout(pad=2.5)
plt.savefig('kmeans_cluster_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Step 5b: 3D Cluster Snapshot at a Sample Timestamp ──
# Red stars (★) = elected Cluster Heads.

# from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# t_sample       = min_timesteps // 2
# labels_sample  = cluster_labels_all[t_sample]
# positions_samp = position_matrix[t_sample]
# heads_sample   = cluster_heads_all[t_sample]

# cmap = plt.cm.get_cmap('tab20', OPTIMAL_K)

# fig = plt.figure(figsize=(13, 9))
# ax  = fig.add_subplot(111, projection='3d')

# for cid in range(OPTIMAL_K):
#     mask = labels_sample == cid
#     ax.scatter(
#         positions_samp[mask, 1],   # longitude → X
#         positions_samp[mask, 0],   # latitude  → Y
#         positions_samp[mask, 2],   # height    → Z
#         c=[cmap(cid)], label=f'C{cid}', s=18, alpha=0.75
#     )

# # Mark cluster heads as red stars
# for cid, uav_global_idx in heads_sample.items():
#     li = uav_ids.index(uav_global_idx)
#     ax.scatter(
#         positions_samp[li, 1], positions_samp[li, 0], positions_samp[li, 2],
#         c='red', s=200, marker='*', zorder=10, label='_nolegend_'
#     )

# ax.set_xlabel('Longitude', fontsize=11)
# ax.set_ylabel('Latitude',  fontsize=11)
# ax.set_zlabel('Height',    fontsize=11)
# ax.set_title(
#     f'3D UAV Cluster Snapshot — t={t_sample}   K={OPTIMAL_K}\n'
#     f'(★ = Cluster Head  |  Every UAV is assigned to a cluster)',
#     fontsize=13
# )
# handles, lbls = ax.get_legend_handles_labels()
# ax.legend(handles, lbls, loc='upper left', fontsize=7, ncol=3)
# plt.tight_layout()
# plt.savefig('3d_kmeans_cluster_snapshot.png', dpi=150, bbox_inches='tight')
# plt.show()

# ── Step 5b: Side-by-side 3D + 2D Cluster Snapshot ──
# Left:  improved 3D scatter with filled floor hulls
# Right: 2D top-down with filled convex-hull regions, marker size ∝ height

# ── Step 5b: 3D Cluster Snapshot at a Sample Timestamp ──
# Red/gold stars = elected Cluster Heads.

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.spatial import ConvexHull

# ─── 1. Pick the snapshot timestep ───────────────────────────────────
t_sample       = min_timesteps // 2
labels_sample  = cluster_labels_all[t_sample]
positions_samp = position_matrix[t_sample]
heads_sample   = cluster_heads_all[t_sample]

# ─── 2. Vivid, well-separated palette (hand-picked) ──────────────────
palette = [
    '#E63946',   # vivid red
    '#06A77D',   # green
    '#F4A261',   # warm orange
    '#7209B7',   # deep purple
    '#118AB2',   # mid blue
    '#073B4C',   # near-black navy
    '#9D0208', '#FB8500', '#0077B6', '#6A040F',   # extras if K > 6
]

# ─── 3. Figure setup ─────────────────────────────────────────────────
fig = plt.figure(figsize=(12, 9), dpi=150)
ax  = fig.add_subplot(111, projection='3d')

z_floor = positions_samp[:, 2].min() - 0.5

# ─── 4. Plot each cluster ────────────────────────────────────────────
for cid in range(OPTIMAL_K):
    mask  = labels_sample == cid
    pts   = positions_samp[mask]
    color = palette[cid % len(palette)]

    # UAV scatter points (bold, edged, opaque)
    ax.scatter(
        pts[:, 1], pts[:, 0], pts[:, 2],
        c=color, s=42, alpha=0.85,
        edgecolors='black', linewidths=0.4,
        label=f'Cluster {cid}'
    )

    # Filled floor hull — projects each cluster's lat/lon footprint
    if len(pts) >= 4:
        try:
            hull = ConvexHull(pts[:, [1, 0]])  # (lon, lat)
            hull_pts = pts[hull.vertices][:, [1, 0]]
            verts = [list(zip(hull_pts[:, 0], hull_pts[:, 1],
                              [z_floor] * len(hull_pts)))]
            poly = Poly3DCollection(verts, alpha=0.18,
                                    facecolor=color, edgecolor=color,
                                    linewidth=1.4)
            ax.add_collection3d(poly)
        except Exception:
            pass

# ─── 5. Cluster heads — same color as cluster, gold-edged stars ──────
for cid, uav_global_idx in heads_sample.items():
    li    = uav_ids.index(uav_global_idx)
    color = palette[cid % len(palette)]

    ax.scatter(
        positions_samp[li, 1], positions_samp[li, 0], positions_samp[li, 2],
        c=color, s=420, marker='*',
        edgecolors='gold', linewidths=2.2, zorder=20
    )

# ─── 6. Cosmetics ────────────────────────────────────────────────────
ax.set_xlabel('Longitude', fontsize=12, labelpad=10)
ax.set_ylabel('Latitude',  fontsize=12, labelpad=10)
ax.set_zlabel('Height (m)', fontsize=12, labelpad=10)
ax.set_title(
    f'3D UAV Cluster Snapshot at $t={t_sample}$, $K={OPTIMAL_K}$\n'
    r'($\bigstar$ = elected cluster head)',
    fontsize=13, pad=15
)
ax.view_init(elev=18, azim=-65)            # tilted view so height reads
ax.grid(alpha=0.3)
ax.xaxis.pane.set_alpha(0.05)              # lighter background panes
ax.yaxis.pane.set_alpha(0.05)
ax.zaxis.pane.set_alpha(0.05)

ax.legend(loc='upper left', fontsize=9, ncol=2, framealpha=0.9)

# ─── 7. Save & show ──────────────────────────────────────────────────
plt.tight_layout()
plt.savefig('3d_kmeans_cluster_snapshot.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# ── Step 6: Export Full Clustering Results to CSV ──
#
# Columns:
#   timestamp       — time index (0-based)
#   uav_id          — UAV identifier (UAV_001 … UAV_500)
#   pred_lat        — LSTM-predicted latitude
#   pred_lon        — LSTM-predicted longitude
#   pred_height     — height at that timestep
#   cluster_id      — K-Means cluster label (0 … K-1)
#   is_cluster_head — True if this UAV is the elected head of its cluster

head_sets = [set(h.values()) for h in cluster_heads_all]

rows = []
for t in range(min_timesteps):
    labels_t = cluster_labels_all[t]
    heads_t  = head_sets[t]
    for local_idx, uav_global_idx in enumerate(uav_ids):
        cid              = int(labels_t[local_idx])
        lat, lon, height = position_matrix[t, local_idx]
        is_head          = uav_global_idx in heads_t
        rows.append({
            'timestamp'      : t,
            'uav_id'         : f'UAV_{uav_global_idx + 1:03d}',
            'pred_lat'       : round(float(lat),    7),
            'pred_lon'       : round(float(lon),    7),
            'pred_height'    : round(float(height), 4),
            'cluster_id'     : cid,
            'is_cluster_head': bool(is_head)
        })

df_clusters = pd.DataFrame(rows)
df_clusters.to_csv('uav_dynamic_clusters_kmeans.csv', index=False)

print(f"Saved {len(df_clusters):,} rows  →  uav_dynamic_clusters_kmeans.csv")
print(f"\nSample output (t=0, first 12 rows):")
print(df_clusters[df_clusters['timestamp'] == 0].head(12).to_string(index=False))

# ── Summary stats ──
total_heads = df_clusters[df_clusters['is_cluster_head']].shape[0]
avg_size    = cluster_sizes_all.mean()
load_std    = cluster_sizes_all.std(axis=1).mean()   # avg load imbalance per timestamp

print(f"\n{'='*54}")
print(f"  Final Clustering Summary  (K-Means, K={OPTIMAL_K})")
print(f"{'='*54}")
print(f"  Timesteps                    : {min_timesteps}")
print(f"  UAVs per timestamp           : {n_uavs}")
print(f"  Fixed K                      : {OPTIMAL_K}")
print(f"  Avg UAVs per cluster         : {avg_size:.1f}")
print(f"  Avg load imbalance (std/t)   : {load_std:.2f} UAVs")
print(f"  Total cluster-head events    : {total_heads:,}")
print(f"  Avg inertia (cluster quality): {inertia_per_t.mean():.2f}")
print(f"{'='*54}")
